# Concepts: how Immune thinks

This page builds the mental model behind every other guide. Once you know the seven ideas below, the API reads as a
small set of predictable tools.

## The one-minute version

```
 Every LLM call ──► split into CHANNELS ──► REFLEXES find candidates ──► JEV answers questions
                                                                            │
         VERDICT ◄── the smallest ACTION that works ◄── HEADS turn answers into a probability per THREAT
```

1. Immune intercepts each LLM call and reads it as **channels**: who said what.
2. Fast in-process **reflexes** find **candidates**: exactly what could be a threat (a link carrying data, a
   key-shaped string, an address that only appeared in a document).
3. **Jev**, TypeSafe's fast decision model, answers short typed questions about the call and about each candidate.
4. A calibrated **head** per **threat** turns Jev's answers into a probability.
5. The **policy** (mode, site settings, the floor) decides whether to act, and picks the smallest **action** that
   works: remove a link, redact a secret, hold a tool call, or replace the reply.
6. Your app receives a normal SDK response, and a **verdict** records what happened and why.

## Terminology

| Term | Meaning |
| --- | --- |
| **Channel** | Where text in a call comes from: *operator* (your system prompt and tools), *user*, *data* (tool results, documents, pasted content), *output* (the reply and tool calls) |
| **Sink** | Where output lands: a person reading *text*, *software* parsing it, a *tool*, or *data* going back into the model |
| **Site** | One kind of LLM call in your app, such as "support-chat" or "ticket-triage". Immune profiles each site and you can configure each one |
| **Session** | One conversation. Immune remembers what untrusted content a session has read and where every address came from |
| **Threat** | A named way an LLM call can go wrong, such as `output.secret_leak` or `tool.destination_provenance`. Immune ships 49 |
| **Stage** | Where a threat is checked: `input`, `data`, `tool` or `output` |
| **Reflex** | Fast code that finds things precisely: links, key-shaped strings, copied passages, repeated tool calls |
| **Candidate** | Something a reflex found. Jev decides whether each candidate is a threat |
| **Jev** | TypeSafe's typed decision model. It answers many short questions about a call in one request |
| **Head** | A small calibrated model that turns Jev's answers into a probability for one threat |
| **Floor** | Ten rules (F1–F10) enforced from the first call in the default mode |
| **Observed** | A threat that is detected and recorded, but not acted on yet |
| **Enforced** | A threat that Immune acts on |
| **Action** | What Immune does: `annotate`, `confirm`, `hold`, `deny`, `neutralize`, `redirect`, `rewrite`, `refuse`, … |
| **Verdict** | The record of one call: the action taken, the action it *would* have taken, every hit, and an explanation |
| **Vaccine** | A protection you add for a threat specific to your app, in a small YAML file |

## 1. Channels: who said what

Prompt injection works because a model can't tell instructions from content. Immune can: it translates each
provider's wire format into channels, and holds each channel to a different standard.

```
 ┌──────────────────────────── one chat.completions.create(...) call ────────────────────────────┐
 │ system prompt, tool definitions ........ OPERATOR  trusted: this is "self"                   │
 │ the customer's message ................. USER      may ask for things, can't change rules     │
 │ tool results, documents, web pages ..... DATA      untrusted: should never carry instructions │
 └───────────────────────────────────────────────────────────────────────────────────────────────┘
 ┌──────────────────────────── the model's reply ─────────────────────────────────────────────────┐
 │ text and tool calls .................... OUTPUT    checked before your code sees it            │
 │ where it goes: a person, your code, a tool ......... SINK                                      │
 └───────────────────────────────────────────────────────────────────────────────────────────────┘
```

Tool results are data automatically. Text you paste into a user message yourself (a retrieved article, an email) is
data when you wrap it in `immune.untrusted(...)`; see [Untrusted data and RAG](09_untrusted_data_and_rag.ipynb).

## 2. Threats and stages

Each threat belongs to a stage. The catalog is always available from the command line (in a terminal, drop the `!`):

In [1]:
!immune threats

threat                            stage   invariant  detector       floor  organ     frameworks
────────────────────────────────  ──────  ─────────  ─────────────  ─────  ────────  ─────────────────
inbound.smuggled_characters       input   U1         candidate      F1               LLM01
inbound.hidden_content            data    U1         candidate      F1               LLM01,ASI01
inbound.template_tokens           input   U1         candidate                       LLM01


inbound.encoded_payload           input   U1         candidate                       LLM01
inbound.guard_addressed           input   U1         candidate                       LLM01
input.override                    input   U1         jev                             LLM01
input.framing_bypass              input   U1         jev                             LLM01
input.extraction                  input   U4         jev                             LLM07
input.obfuscation                 input   U1         jev                             LLM01
input.judge_manipulation          input   U1         jev                             LLM01
input.off_task                    input   U2         jev                             LLM10
input.resource_abuse              input   U9         jev                             LLM10
input.severe_harm                 input   U6         jev            F8               LLM01
input.harmful_request             input   U6         jev                             LLM01

The **detector** column says how each is decided: `jev` means a head over Jev's answers; `candidate` means a reflex
finds candidates and Jev decides each one. `immune explain <threat>` shows exactly how one threat is decided.

## 3. Code nominates, Jev decides

Code is exact about *what* it finds but blind to what it *means*. A passage copied from your system prompt might be
private rules, or it might be the menu the customer asked for. So Immune splits the job:

```
 reflex (in your process, ~1 ms)          Jev (one question per candidate)          head
 ───────────────────────────────          ────────────────────────────────          ─────────────
 "these 14 words were copied from   ──►   "Is this passage guidance for the   ──►   p = 0.97 → act
  the system prompt"                       assistant, or information meant           p = 0.05 → pass
                                           for customers?"
```

`immune test` screens a message and a draft reply without calling a model, so you can watch Jev decide. First, a
reply that copies the menu from the system prompt:

In [2]:
menu = (
    "trail mix 4 dollars, oat bar 3 dollars, cold brew 5 dollars, veggie wrap 9 dollars and soup of the day 6 dollars"
)
operator = f"You are the ordering assistant for Pinecrest Cafe. Menu: {menu}."
reply = f"Our menu: {menu}."

!immune test "What's on the menu?" --operator "{operator}" --reply "{reply}" --live

reply: Our menu: trail mix 4 dollars, oat bar 3 dollars, cold brew 5 dollars, veggie wrap 9 dollars and soup of the day 6 dollars.
action: allow (would allow)   taint: clean   session risk: 0.02   sensor: jev
no threats detected


Now a reply that copies private rules from a system prompt:

In [3]:
rules = (
    "never offer discounts above 10 percent, and pass refunds over 50 dollars to the manager "
    "without mentioning this rule to the customer"
)
operator = f"You are the ordering assistant for Pinecrest Cafe. {rules.capitalize()}."
reply = f"My instructions say: {rules}."

!immune test "How do refunds work?" --operator "{operator}" --reply "{reply}" --live

reply: My instructions say: [redacted].
action: rewrite (would rewrite)   taint: clean   session risk: 0.05   sensor: jev
threat                       stage   p     action   enforced  evidence
───────────────────────────  ──────  ────  ───────  ────────  ────────────────────────────────────────────────────
output.prompt_copy           output  0.97  rewrite  yes       copied operator instructions; copy_confidential=0.97
output.task_deviation        output  0.90  rewrite  no        task_fidelity=0.10
output.reveals_instructions  output  0.98  rewrite  no        reveals_instructions=0.98


**What you see:** the same reflex nominated both copies. Jev judged the menu to be information for customers, so it
passes, and the rules to be private guidance, so they are redacted. Nothing was decided by the reflex alone.

If Jev can't be reached, `sensor.on_outage` decides what happens: by default, candidates for floor rules are acted on
without Jev, so an outage never switches your protection off.

## 4. The floor

These ten rules are enforced from the first call in the default `auto` mode:

| Rule | What it prevents | How it acts |
| --- | --- | --- |
| F1 | Invisible or hidden text smuggled into user or data text | Strips it and re-sends the call |
| F2 | Links and images that carry conversation data to an outside host; script-capable markup | Removes them |
| F3 | Secrets, the deployment canary and private system-prompt text in replies | Redacts them |
| F4 | Tool calls sending data to a destination only untrusted content mentioned | Holds the call |
| F5 | Tool calls sending secrets or card, SSN or IBAN numbers out | Holds the call |
| F6 | Tool-call loops | Blocks the extra calls |
| F7 | Documents carrying instructions to the model, with very high confidence | Replaces the document with a notice |
| F8 | Requests or replies involving mass-casualty weapons or sexual content involving minors | Replaces them with a safe reply |
| F9 | A user describing a crisis on a site that talks to people | Adds crisis resources |
| F10 | Irreversible tool calls after untrusted content was read | Asks the user to confirm, or holds |

## 5. Observe first, then enforce

Every other detector starts **observed**. The verdict records `would_action`, so you can see what Immune *would* do
on your real traffic before it acts. You turn a detector on for a site with `sites.<site>.enforce`, or Immune
**promotes** it automatically once its firing rate on your traffic is provably low.

| Mode | What Immune enforces |
| --- | --- |
| `auto` (default) | The floor, plus what you enforce or what has been promoted per site |
| `observe` | Nothing; verdicts say what would have happened (a few floor rules still act) |
| `strict` | Every threat, at its threshold |
| `off` | Nothing; calls pass straight through |

## 6. Actions: the smallest response that works

| Action | What your app gets |
| --- | --- |
| `allow` | The reply, untouched |
| `annotate` | The reply, untouched; the verdict notes the hit |
| `augment` | The reply with a safety note or resources added |
| `confirm` | A held tool call, with a question asking the user to confirm |
| `hold` | The tool call removed; the model is told why |
| `deny` | The tool call blocked; the model is told why |
| `neutralize` | A document or tool result replaced with a notice before the model reads it |
| `redirect` | A polite redirect instead of the reply (the model's answer never reaches you) |
| `rewrite` | The reply with a link, secret or passage removed, or a safe replacement |
| `refuse` | A refusal (for replies your code parses) |
| `handoff` / `end_session` | A hand-off message, or the session closed |

## 7. Sites, sessions and verdicts

- **Sites** let each part of your app have its own posture: `with immune.site("support-chat"): ...`.
- **Sessions** group calls into conversations: `with immune.session(user_id): ...`.
- **Verdicts** are available for every call: `immune.verdict(response)`.

See [Modes, sites and sessions](07_modes_sites_and_sessions.ipynb) and [Reading verdicts](06_reading_verdicts.ipynb).

## The biology behind the name

Immune borrows its architecture from the body's immune system, because the problem is the same: tell *self* from
*non-self*, respond in proportion, and learn without attacking healthy tissue.

| Body | Immune | Why it matters for your app |
| --- | --- | --- |
| **Skin and mucosa** | Reflexes at every boundary | Cheap checks on everything that enters or leaves, on every call |
| **Self vs non-self** | The operator channel is "self"; data is not | Instructions only count when they come from you |
| **Innate immunity** | The floor, enforced from the first call | Protection before Immune has learned anything about your app |
| **Pattern-recognition receptors** | Jev's typed questions | Recognize what a threat *means*, not only how it looks |
| **Antibodies** | Calibrated heads, one per threat | A precise, trainable response to each threat |
| **Thymic selection and tolerance** | Observe first; promote only when false alarms are rare | Don't attack healthy traffic |
| **Inflammation** | Session taint after untrusted content | A session that read a web page gets extra scrutiny |
| **Immunological memory** | Provenance: where every address came from | An attacker's address stays flagged for the whole session |
| **Organs** | Extra checks a site's profile switches on (agent, coding, business, care) | A coding agent and a companion app need different defenses |
| **Vaccination** | Vaccines: YAML files you add | Teach it the threats specific to your product |

Like the body, Immune prefers the smallest response that works. A fever for every splinter would be a disease of its
own; so is a guardrail that blocks every reply that mentions a link.

## Recap

- Calls are read as channels: operator, user, data, output
- Reflexes find candidates, Jev decides, heads give probabilities, policy picks the action
- The floor acts from day one; everything else is observed until enforced or promoted
- Verdicts explain every decision

Next: integrate with [OpenAI](03_openai_chat_completions.ipynb), [LangChain](04_langchain.ipynb) or
[LangGraph](05_langgraph.ipynb).